In [8]:
from pathlib import Path

import numpy as np

from msmjax.benchmark_tools import evaluate_structure_with_lammps_p3m

In [9]:
LAMMPS_EXECUTABLE = "/home/florian/Downloads/lammps-static/bin/lmp"

# The cutoffs used by p3m to get the desired accuracy can be quite large
MAX_NEIGHBORS_ONE_ATOM = 10000

PATH_STRUCTURES = Path("structures/")
OUTDIR_RESULTS = Path("results_ref_periodic_lammps_p3m/")

OUTDIR_RESULTS.mkdir()


FileExistsError: [Errno 17] File exists: 'results_ref_periodic_lammps_p3m'

In [10]:
for n_particles in [50, 100, 200, 300, 400, 500, 1000, 1500, 2000]:
    structures = np.load(PATH_STRUCTURES / f"structures_{n_particles}.npz")
    n_structures = structures["positions"].shape[0]
    
    energies_all_structures = []
    forces_all_structures = []
    for i_structure in range(n_structures):
        pos = structures["positions"][i_structure].astype(np.float64)
        chg = structures["charges"][i_structure].astype(np.float64)
        cell = structures["cells"][i_structure].astype(np.float64)
        energy, forces = evaluate_structure_with_lammps_p3m(
            pos,
            chg,
            cell,
            lammps_executable=LAMMPS_EXECUTABLE,
            max_neighbors_one_atom=MAX_NEIGHBORS_ONE_ATOM,
        )
        energies_all_structures.append(energy)
        forces_all_structures.append(forces)
        
    energies_all_structures = np.array(energies_all_structures)
    forces_all_structures = np.array(forces_all_structures)
    
    np.savez_compressed(
        OUTDIR_RESULTS / f"results_{n_particles}.npz",
        energies=energies_all_structures,
        forces=forces_all_structures,
    )